In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import classification_report, f1_score, roc_auc_score

In [2]:
# 1. Setup: Load Data and Stratified 80/20 Split
df_clean = pd.read_csv('../data/customer_churn_clean.csv')
# Prepare Categorical Variables using One-Hot Encoding
X = pd.get_dummies(df_clean.drop('churn', axis=1), drop_first=True)
y = df_clean['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [3]:
# 2. Model Dictionary
models = {
    "LogisticRegression": LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "RandomForest": RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42)
}

In [4]:
# 3. The Loop
summary = []
for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_test_pred = model.predict(X_test)
    y_test_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_test_pred
    
    # Calculate Metrics
    report = classification_report(y_test, y_test_pred, output_dict=True)
    recall_1 = report['1']['recall']
    f1 = f1_score(y_test, y_test_pred)
    roc = roc_auc_score(y_test, y_test_prob)
    
    summary.append({'Model': name, 'Test Recall': recall_1, 'Test F1-Score': f1, 'ROC AUC': roc})


c:\Users\jaisw\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
summary_df = pd.DataFrame(summary)
print("--- PHASE 2: BASELINE ZERO-TUNING RESULTS ---")
print(summary_df.to_string(index=False))

--- PHASE 2: BASELINE ZERO-TUNING RESULTS ---
             Model  Test Recall  Test F1-Score  ROC AUC
LogisticRegression     0.642157       0.300804 0.753163
      DecisionTree     0.166667       0.163855 0.534057
      RandomForest     0.000000       0.000000 0.778452
          AdaBoost     0.151961       0.238462 0.805868
